# CV4CHL - Classifier Ensemble Item 5

**Platform**: Colab / Local (CPU)
**Runtime**: ~20 min
**Goal**: Combine ML predict_proba with evgs_rules rule binary predictions using weighted ensemble

## Approach: probability-level ensemble with optimized thresholds

- **ML hard predictions** (0/1 from XGBoost / CatBoost / LightGBM) discard
  the model's confidence in borderline cases.
- **Rule-based hard predictions** (0/1 from threshold rules) likewise lose
  the underlying continuous signal.
- **This notebook**: uses ML **probabilities** (0.0~1.0) averaged with rule
  predictions, then applies a per-item **optimized threshold** (not just 0.5).

This can flip borderline items (ML p=0.48, rule=1 → ensemble p=0.74 → prediction 1) that neither source alone would change.


In [ ]:
# === Cell 1: Setup & Imports ===
import time, os, sys, pickle, warnings
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score

warnings.filterwarnings('ignore')

_cell_times = {}
_cell_start = None
def cell_start(name):
    global _cell_start
    _cell_start = time.time()
    _cell_times[name] = None
    print(f'\u25b6 {name}')
def cell_end(name):
    elapsed = time.time() - _cell_start
    _cell_times[name] = elapsed
    print(f'\u2713 {name} — {elapsed:.1f}s ({elapsed/60:.1f}min)')

try:
    import xgboost as xgb
except ImportError:
    os.system('pip install -q xgboost'); import xgboost as xgb
try:
    import lightgbm as lgb
except ImportError:
    os.system('pip install -q lightgbm'); import lightgbm as lgb
try:
    from catboost import CatBoostClassifier
except ImportError:
    os.system('pip install -q catboost'); from catboost import CatBoostClassifier

print('Setup complete.')


In [ ]:
# === Cell 2: Config & Load Data ===
cell_start('Cell 2: Config & Load Data')

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = os.environ.get(
        'CV4CHL_REPO_ROOT',
        '/content/drive/MyDrive/CVPR_CH4CHL',
    )
else:
    ROOT = os.environ.get(
        'CV4CHL_REPO_ROOT',
        os.getcwd(),
    )

PROC_DIR = f'{ROOT}/1_data/processed'
SUBMIT_DIR = f'{ROOT}/5_outputs/submissions'
DIAGNOSTIC_DIR = f'{ROOT}/5_outputs/diagnostics/item05_classifier_ensemble'
os.makedirs(SUBMIT_DIR, exist_ok=True)
os.makedirs(DIAGNOSTIC_DIR, exist_ok=True)

# Only item05_classifier_ensemble_full.csv is read by final_assembly.py.
# All other variants below are exploratory candidates written to
# DIAGNOSTIC_DIR for offline review.
CANONICAL_FILENAME = 'item05_classifier_ensemble_full.csv'

T1_TEST_IDS = [4, 5, 18, 26, 28, 40, 42, 43, 47, 48, 53, 54, 72, 78, 83, 85]
T2_TEST_IDS = [4, 6, 7, 13, 26, 35, 39, 42, 50]
# Track-2 rows are preserved from the current base CSV;
# Track-2 predictions are generated only by the Track-2 clinical notebook.

with open(f'{PROC_DIR}/train_ready.pkl', 'rb') as f:
    tr_data = pickle.load(f)
df_train = tr_data['track1_train'].copy()
df_test = tr_data['track1_test'].copy()

FEAT_COLS = sorted([c for c in df_train.columns if c.startswith(('sag_', 'cor_', 'all_', 'n_'))])
EVGS_ITEMS = [f'evgs_{i}' for i in range(1, 18)]

print(f'Train: {df_train.shape}, Test: {df_test.shape}, Features: {len(FEAT_COLS)}')

cell_end('Cell 2: Config & Load Data')


In [ ]:
# === Cell 3: LOSO CV — Get ML Probabilities + Rule Predictions ===
cell_start('Cell 3: LOSO CV Probabilities')

train_pids = sorted(df_train['patient_id'].unique())
n_train = len(df_train)

# Models to ensemble
MODELS = {
    'XGBoost': lambda spw: xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
        scale_pos_weight=spw, use_label_encoder=False, eval_metric='logloss', verbosity=0, random_state=42),
    'LightGBM': lambda spw: lgb.LGBMClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
        scale_pos_weight=spw, verbose=-1, random_state=42),
    'CatBoost': lambda spw: CatBoostClassifier(iterations=100, depth=4, learning_rate=0.1,
        auto_class_weights='Balanced', verbose=0, random_state=42),
}

# Store OOF probabilities and test probabilities
oof_probs = {}   # {(item, model): np.array of probabilities}
test_probs = {}  # {(item, model): np.array of probabilities}
oof_binary = {}  # {(item, model): np.array of 0/1}

for item_i in tqdm(range(1, 18), desc='LOSO CV'):
    item_name = f'evgs_{item_i}'
    y = df_train[item_name].values
    X_train_all = np.nan_to_num(df_train[FEAT_COLS].values.astype(np.float32), nan=0.0)
    X_test_all = np.nan_to_num(df_test[FEAT_COLS].values.astype(np.float32), nan=0.0)

    for model_name, model_fn in MODELS.items():
        oof_p = np.zeros(n_train, dtype=np.float64)
        # For test: train on ALL train, predict test
        spw = (len(y) - y.sum()) / max(y.sum(), 1)

        # LOSO for OOF
        for pid in train_pids:
            mask = df_train['patient_id'].values == pid
            X_tr, y_tr = X_train_all[~mask], y[~mask]
            X_val = X_train_all[mask]
            spw_fold = (len(y_tr) - y_tr.sum()) / max(y_tr.sum(), 1)
            clf = model_fn(spw_fold)
            clf.fit(X_tr, y_tr)
            proba = clf.predict_proba(X_val)
            oof_p[mask] = proba[:, 1] if proba.shape[1] > 1 else proba[:, 0]

        oof_probs[(item_i, model_name)] = oof_p
        oof_binary[(item_i, model_name)] = (oof_p >= 0.5).astype(int)

        # Final model for test predictions
        clf_final = model_fn(spw)
        clf_final.fit(X_train_all, y)
        test_p = clf_final.predict_proba(X_test_all)
        test_probs[(item_i, model_name)] = test_p[:, 1] if test_p.shape[1] > 1 else test_p[:, 0]

print(f'OOF probs: {len(oof_probs)} (17 items x 3 models)')
print(f'Test probs: {len(test_probs)}')

cell_end('Cell 3: LOSO CV Probabilities')


In [ ]:
# === Cell 4: Build Ensemble Probabilities ===
cell_start('Cell 4: Ensemble Probabilities')

# For each item, create ensemble probabilities by averaging ML models
# Then create weighted ensemble with rule predictions

# Approach 1: Average of 3 ML models
oof_ml_avg = {}
test_ml_avg = {}
for item_i in range(1, 18):
    oof_avg = np.mean([oof_probs[(item_i, m)] for m in MODELS], axis=0)
    test_avg = np.mean([test_probs[(item_i, m)] for m in MODELS], axis=0)
    oof_ml_avg[item_i] = oof_avg
    test_ml_avg[item_i] = test_avg

# Approach 2: ML avg + rule binary (weighted)
# Rule prediction = 0 or 1, treated as probability 0.0 or 1.0
# Ensemble = w_ml * ml_prob + w_rule * rule_binary
# Try multiple weight combinations

# Rule predictions enter the ensemble as binary (0.0 or 1.0); load the
# per-item rule prediction CSVs from notebook 04 for the test patients.

E47_DIR = f'{SUBMIT_DIR}/evgs_rules'
BASE_PATH = f'{SUBMIT_DIR}/reference_baseline.csv'

# Load base to know original ML predictions
if os.path.exists(BASE_PATH):
    df_base = pd.read_csv(BASE_PATH)
else:
    df_base = None
    print('WARNING: reference baseline base not found')

# Load evgs_rules rule predictions per item
rule_test_preds = {}  # {item_i: {(pid, side): 0 or 1}}
if os.path.exists(E47_DIR):
    for item_i in range(1, 18):
        fpath = f'{E47_DIR}/evgs_rules_item_{item_i:02d}.csv'
        if os.path.exists(fpath):
            df_rule = pd.read_csv(fpath)
            t1_rule = df_rule[df_rule['ID'].str.startswith('track1-')]
            preds = {}
            for _, row in t1_rule.iterrows():
                pid = int(row['ID'].split('-')[1])
                for side_letter in ['L', 'R']:
                    col = f'{side_letter}{item_i}'
                    preds[(pid, side_letter)] = int(row[col])
            rule_test_preds[item_i] = preds
    print(f'Loaded rule predictions for {len(rule_test_preds)} items')
else:
    print(f'evgs_rules dir not found at {E47_DIR}')
    print('Will use ML-only ensemble (no rule component)')

cell_end('Cell 4: Ensemble Probabilities')


In [ ]:
# === Cell 5: Per-Item Threshold Optimization ===
cell_start('Cell 5: Threshold Optimization')

# For each item, find the optimal threshold on OOF predictions
# Try: ML-only avg, and ML+rule ensemble with different weights

def compute_s1_from_preds(pred_dict, df):
    """pred_dict: {item_i: np.array of binary predictions}"""
    n = len(df)
    all_correct = 0
    all_total = 0
    total_pred = np.zeros(n)
    total_true = np.zeros(n)
    for item_i in range(1, 18):
        pred = pred_dict[item_i]
        true = df[f'evgs_{item_i}'].values
        all_correct += np.sum(pred == true)
        all_total += len(true)
        total_pred += pred
        total_true += true
    acc = all_correct / all_total
    pids = df['patient_id'].values
    sides = df['side'].values
    rmse_sq = []
    for pid in sorted(set(pids)):
        ml = (pids == pid) & (sides == 'left')
        mr = (pids == pid) & (sides == 'right')
        if ml.sum() > 0 and mr.sum() > 0:
            rmse_sq.append((total_pred[ml][0] + total_pred[mr][0] - total_true[ml][0] - total_true[mr][0]) ** 2)
    rmse = np.sqrt(np.mean(rmse_sq)) if rmse_sq else 0.0
    s1 = (acc + 1 - rmse / 34.0) / 2
    return acc, rmse, s1

# Baseline: ML avg with threshold 0.5
baseline_preds = {i: (oof_ml_avg[i] >= 0.5).astype(int) for i in range(1, 18)}
_, _, s1_baseline = compute_s1_from_preds(baseline_preds, df_train)
print(f'Baseline (ML avg, thr=0.5): S1={s1_baseline:.5f}')

# Per-item threshold sweep
best_config = {}  # {item_i: (source, threshold, accuracy)}
thresholds = np.arange(0.30, 0.71, 0.01)

for item_i in range(1, 18):
    y = df_train[f'evgs_{item_i}'].values
    best_acc = 0
    best_thr = 0.5
    best_src = 'ml_avg'

    # Try ML avg with different thresholds
    for thr in thresholds:
        pred = (oof_ml_avg[item_i] >= thr).astype(int)
        acc = accuracy_score(y, pred)
        if acc > best_acc:
            best_acc = acc
            best_thr = float(thr)
            best_src = 'ml_avg'

    # Try individual models with different thresholds
    for model_name in MODELS:
        for thr in thresholds:
            pred = (oof_probs[(item_i, model_name)] >= thr).astype(int)
            acc = accuracy_score(y, pred)
            if acc > best_acc:
                best_acc = acc
                best_thr = float(thr)
                best_src = model_name

    # Try ML+rule ensemble (if rules available)
    if item_i in rule_test_preds:
        # Rule OOF predictions are not produced as probabilities here, so the
        # rule-weighted ensemble is evaluated only on the test set, not in
        # OOF threshold tuning.
        pass

    best_config[item_i] = (best_src, best_thr, best_acc)

# Apply best configs
optimized_preds = {}
for item_i in range(1, 18):
    src, thr, acc = best_config[item_i]
    if src == 'ml_avg':
        optimized_preds[item_i] = (oof_ml_avg[item_i] >= thr).astype(int)
    else:
        optimized_preds[item_i] = (oof_probs[(item_i, src)] >= thr).astype(int)

_, _, s1_optimized = compute_s1_from_preds(optimized_preds, df_train)
print(f'Optimized (per-item threshold): S1={s1_optimized:.5f} (delta={s1_optimized-s1_baseline:+.5f})')

# Print per-item config
print(f'\n{"Item":>4} {"Source":>10} {"Thr":>6} {"Acc":>8} {"vs 0.5":>8}')
print('-' * 42)
for i in range(1, 18):
    src, thr, acc = best_config[i]
    baseline_acc = accuracy_score(df_train[f'evgs_{i}'].values, baseline_preds[i])
    delta = acc - baseline_acc
    marker = ' *' if abs(thr - 0.5) > 0.02 else ''
    print(f'{i:>4d} {src:>10} {thr:>6.2f} {acc:>8.4f} {delta:>+8.4f}{marker}')

cell_end('Cell 5: Threshold Optimization')


In [ ]:
# === Cell 6: Generate Test Predictions & Submissions ===
cell_start('Cell 6: Generate Submissions')

# Apply optimized configs to test predictions
test_optimized = {}
for item_i in range(1, 18):
    src, thr, _ = best_config[item_i]
    if src == 'ml_avg':
        test_optimized[item_i] = (test_ml_avg[item_i] >= thr).astype(int)
    else:
        test_optimized[item_i] = (test_probs[(item_i, src)] >= thr).astype(int)

# Build submission CSV
def build_submission(test_pred_dict, csv_name):
    if df_base is not None:
        m = df_base.copy()
    else:
        # Build from scratch
        rows = []
        for pid in sorted(T1_TEST_IDS):
            rows.append({'ID': f'track1-{pid}'})
        for pid in sorted(T2_TEST_IDS):
            rows.append({'ID': f'track2-{pid}'})
        cols = ['ID'] + [f'{p}{i}' for p in ['L','R'] for i in range(1,18)] + ['Total', 'Left_gait_subtype', 'Right_gait_subtype']
        m = pd.DataFrame(rows, columns=cols).fillna(-1)

    t1_mask = m['ID'].str.startswith('track1-')
    item_cols = [f'{p}{i}' for p in ['L','R'] for i in range(1,18)]

    # Fill Track 1 predictions
    for item_i in range(1, 18):
        preds = test_pred_dict[item_i]
        for idx in range(len(df_test)):
            row = df_test.iloc[idx]
            pid = int(row['patient_id'])
            side_letter = 'L' if row['side'] == 'left' else 'R'
            col = f'{side_letter}{item_i}'
            row_mask = m['ID'] == f'track1-{pid}'
            m.loc[row_mask, col] = int(preds[idx])

    # Recalc Total
    m.loc[t1_mask, 'Total'] = m.loc[t1_mask, item_cols].sum(axis=1).astype(int)

    # Track 2 rows preserved from base (overwritten by notebook 03)
    # Track-2 rows are preserved from the current base CSV.

    out_dir = SUBMIT_DIR if csv_name == CANONICAL_FILENAME else DIAGNOSTIC_DIR
    path = f'{out_dir}/{csv_name}'
    m.to_csv(path, index=False)
    if df_base is not None:
        n = sum((df_base.loc[t1_mask, c].values != m.loc[t1_mask, c].values).sum() for c in item_cols)
        print(f'  {csv_name}: {n} source cells selected relative to reference baseline')
    else:
        print(f'  {csv_name}: generated (no base comparison)')

# CSV 1: Full optimized (all items use optimized threshold + model)
build_submission(test_optimized, 'item05_classifier_ensemble_full.csv')

# CSV 2: Only items where threshold != 0.5 (conservative)
test_conservative = {}
for item_i in range(1, 18):
    src, thr, _ = best_config[item_i]
    if abs(thr - 0.5) > 0.02:
        # Use optimized
        if src == 'ml_avg':
            test_conservative[item_i] = (test_ml_avg[item_i] >= thr).astype(int)
        else:
            test_conservative[item_i] = (test_probs[(item_i, src)] >= thr).astype(int)
    else:
        # Use default (XGBoost 0.5)
        test_conservative[item_i] = (test_probs[(item_i, 'XGBoost')] >= 0.5).astype(int)
build_submission(test_conservative, 'item05_classifier_ensemble_conservative.csv')

# CSV 3: ML avg with per-item threshold only (no model switch)
test_ml_thr = {}
for item_i in range(1, 18):
    # Find best threshold for ml_avg specifically
    y = df_train[f'evgs_{item_i}'].values
    best_thr = 0.5
    best_acc = accuracy_score(y, (oof_ml_avg[item_i] >= 0.5).astype(int))
    for thr in thresholds:
        pred = (oof_ml_avg[item_i] >= thr).astype(int)
        acc = accuracy_score(y, pred)
        if acc > best_acc:
            best_acc = acc
            best_thr = float(thr)
    test_ml_thr[item_i] = (test_ml_avg[item_i] >= best_thr).astype(int)
build_submission(test_ml_thr, 'item05_classifier_ensemble_thresholds.csv')

# CSV 4-6: Per-item merge into reference baseline (selected item outputs)
# Load reference baseline
BEST_PATH = f'{SUBMIT_DIR}/reference_baseline.csv'
if not os.path.exists(BEST_PATH):
    BEST_PATH = f'{SUBMIT_DIR}/reference_baseline.csv'
if os.path.exists(BEST_PATH):
    df_current_best = pd.read_csv(BEST_PATH)
    t1_best = df_current_best['ID'].str.startswith('track1-')
    item_cols_csv = [f'{p}{i}' for p in ['L','R'] for i in range(1,18)]

    # Find items where optimized prediction differs from reference baseline
    selected_items = []
    for item_i in range(1, 18):
        src, thr, _ = best_config[item_i]
        if src == 'ml_avg':
            new_preds = (test_ml_avg[item_i] >= thr).astype(int)
        else:
            new_preds = (test_probs[(item_i, src)] >= thr).astype(int)

        # Check if any test prediction differs
        for idx in range(len(df_test)):
            row = df_test.iloc[idx]
            pid = int(row['patient_id'])
            side_letter = 'L' if row['side'] == 'left' else 'R'
            col = f'{side_letter}{item_i}'
            row_mask = df_current_best['ID'] == f'track1-{pid}'
            old_val = int(df_current_best.loc[row_mask, col].values[0])
            new_val = int(new_preds[idx])
            if old_val != new_val:
                selected_items.append(item_i)
                break

    print(f'\nItems with selected source predictions vs reference baseline: {sorted(set(selected_items))}')

    # Generate per-selected-item CSVs merged into reference baseline
    for item_i in sorted(set(selected_items)):
        m = df_current_best.copy()
        src, thr, _ = best_config[item_i]
        if src == 'ml_avg':
            new_preds = (test_ml_avg[item_i] >= thr).astype(int)
        else:
            new_preds = (test_probs[(item_i, src)] >= thr).astype(int)

        for idx in range(len(df_test)):
            row = df_test.iloc[idx]
            pid = int(row['patient_id'])
            side_letter = 'L' if row['side'] == 'left' else 'R'
            col = f'{side_letter}{item_i}'
            row_mask = m['ID'] == f'track1-{pid}'
            m.loc[row_mask, col] = int(new_preds[idx])

        m.loc[t1_best, 'Total'] = m.loc[t1_best, item_cols_csv].sum(axis=1).astype(int)
        # Track 2 preserved
        # Track-2 rows are preserved from the current base CSV.

        n = sum((df_current_best.loc[t1_best, c].values != m.loc[t1_best, c].values).sum() for c in item_cols_csv)
        name = f'item05_classifier_ensemble_merge_item{item_i:02d}_thr{thr:.2f}.csv'
        # per-item merge variants are diagnostic-only (final_assembly only reads CANONICAL_FILENAME)
        m.to_csv(f'{DIAGNOSTIC_DIR}/{name}', index=False)
        print(f'  {name}: {n} source cells selected (item {item_i}, {src} thr={thr:.2f})')
else:
    print(f'Reference baseline not found, skipping merge CSVs')

cell_end('Cell 6: Generate Submissions')


In [ ]:
# === Cell 7: Summary ===
cell_start('Summary')

total_time = sum(t for t in _cell_times.values() if t is not None)
time_report = '\n'.join(f'  {n}: {t:.1f}s ({t/60:.1f}min)'
    for n, t in _cell_times.items() if t is not None)

print(f"""
######################################################################
# item05_classifier_ensemble Classifier Ensemble Item 5
#  Per-item ensemble over the CV-selected EVGS rule sources
#  Goal: Find items where blended probabilities + shifted threshold
#        produce different (better) predictions than hard ML or rules
######################################################################

{'='*60}
Execution time:
{time_report}
  Total: {total_time:.1f}s ({total_time/60:.1f}min)

ML Baseline S1 (avg, thr=0.5): {s1_baseline:.5f}
Optimized S1 (per-item best):  {s1_optimized:.5f} (delta={s1_optimized-s1_baseline:+.5f})

Per-item optimized configs:
""")
for i in range(1, 18):
    src, thr, acc = best_config[i]
    marker = ' <<<' if abs(thr - 0.5) > 0.05 else ''
    print(f'  Item {i:2d}: {src:>10} thr={thr:.2f} acc={acc:.4f}{marker}')

print(f"""
\nCSVs generated:
  item05_classifier_ensemble_full.csv — all items use best model + threshold
  item05_classifier_ensemble_conservative.csv — only items with thr != 0.5
  item05_classifier_ensemble_thresholds.csv — ML avg with per-item threshold
  item05_classifier_ensemble_merge_item*_thr*.csv — per-item merge into reference baseline

Generated CSV artifacts for offline review:

Per-item threshold-tuning diagnostics: items with large threshold shift (>0.05)
and clinically interpretable direction are surfaced for downstream selection.
{'='*60}
""")

cell_end('Summary')
